In [18]:
import pandas as pd
df = pd.read_csv('internship_selection_prediction.csv')

# This will print every column name
print(df.columns.tolist())

# This will show the first 2 rows
print(df.head(2))

['student_id', 'CGPA', 'skills_score', 'projects_count', 'internships_done', 'communication_score', 'aptitude_score', 'coding_test_score', 'resume_score', 'extracurricular', 'college_tier', 'hackathons_participated', 'certifications_count', 'linkedin_activity_score', 'github_score', 'soft_skills_score', 'interview_score', 'consistency_score', 'backlogs', 'placement_training', 'selected']
   student_id  CGPA  skills_score  projects_count  internships_done  \
0           1  6.87             7               0                 0   
1           2  9.75             4               4                 2   

   communication_score  aptitude_score  coding_test_score  resume_score  \
0                    4               3                  2             7   
1                    3               3                  6             1   

  extracurricular  ... hackathons_participated  certifications_count  \
0             Yes  ...                       0                     7   
1             Yes  ...   

In [41]:
print(df.dtypes)
print(df.isnull().sum())
print(df.describe().round(2))

student_id                   int64
CGPA                       float64
skills_score                 int64
projects_count               int64
internships_done             int64
communication_score          int64
aptitude_score               int64
coding_test_score            int64
resume_score                 int64
extracurricular             object
college_tier                object
hackathons_participated      int64
certifications_count         int64
linkedin_activity_score      int64
github_score                 int64
soft_skills_score            int64
interview_score              int64
consistency_score            int64
backlogs                     int64
placement_training          object
selected                     int64
total_soft_skills            int64
Technical_Avg              float64
Behavioral_Avg             float64
Tech_Category               object
Beh_Category                object
dtype: object
student_id                 0
CGPA                       0
skills_score      

Question 1: Distribution & Balance
Is there a balanced or imbalanced distribution of the outcome variable that might affect model bias?

In [19]:
# 1. Check Balance of the 'selected' column
target_counts = df['selected'].value_counts(normalize=True) * 100
print("Percentage of students selected vs rejected:")
print(target_counts)

Percentage of students selected vs rejected:
selected
1    73.74
0    26.26
Name: proportion, dtype: float64


In [20]:
fig_balance = px.pie(df, names='selected', 
             title='Balance of Internship Selection (1=Selected, 0=Rejected)',
             hole=0.4,
             color='selected',
             color_discrete_map={0: 'salmon', 1: 'mediumseagreen'})
fig_balance.show()

### Analysis: 
This shows the competitiveness of the program. If "Selected" is a small slice, the criteria are exclusive. If it's near 50%, the dataset is balanced for fair analysis.

### How to look at it: 
Compare the size of the "Selected" (1) vs. "Rejected" (0) slices.

In [21]:
# 2. Check CGPA Distribution
fig_cgpa_dist = px.histogram(df, x='CGPA', 
                             nbins=20, 
                             title='Distribution of CGPA across all candidates',
                             color_discrete_sequence=['skyblue'])
fig_cgpa_dist.show()

Question 2: CGPA as a Separator
How well is CGPA able to separate selected students from non-selected students?

In [22]:
# Box plot to see the median and outliers for both groups
fig_sep = px.box(df, x='selected', y='CGPA', 
                 points="all", 
                 title='CGPA Separation: Selected vs Rejected',
                 color='selected',
                 color_discrete_map={0: 'salmon', 1: 'mediumseagreen'})
fig_sep.show()

### Analysis: 
This identifies the Academic Floor. If the boxes overlap heavily, it proves that CGPA is not the only factor and "low-grade" students still have a path to success.

### How to look at it: 
Check if the "Selected" box is physically higher than the "Rejected" box.

Question 3: What drives selection?
Calculate correlation with 'selected'

In [25]:
import plotly.figure_factory as ff
import numpy as np

# 1. Prepare numerical data (exclude student_id)
df_numeric = df.drop(columns=['student_id']).select_dtypes(include=[np.number])

# 2. Calculate the correlation matrix
corr_matrix = df_numeric.corr()

# 3. Create the Annotated Heatmap
fig_corr = ff.create_annotated_heatmap(
    z=corr_matrix.values,
    x=list(corr_matrix.columns),
    y=list(corr_matrix.index),
    annotation_text=corr_matrix.round(2).values,
    colorscale='RdBu', # Red for negative, Blue for positive correlation
    zmin=-1, zmax=1
)

fig_corr.update_layout(
    title_text='Correlation Heatmap: Selection Drivers',
    xaxis_tickangle=45
)
fig_corr.show()

# 4. Display the top predictors for 'selected'
print("Top 5 Positive Predictors for Selection:")
print(corr_matrix['selected'].sort_values(ascending=False).iloc[1:6])

Top 5 Positive Predictors for Selection:
interview_score        0.069152
skills_score           0.058453
communication_score    0.044100
soft_skills_score      0.041482
resume_score           0.036477
Name: selected, dtype: float64


### Analysis: 
This ranks the Impact of Traits. If coding_score has a higher number than CGPA, your thesis that "Skills > Grades" is statistically confirmed.

### How to look at it: 
Find the "selected" row and look for the highest numbers (closest to 1.0).

Question 4: College Tier & Institutional Prestige
Does coming from a 'Tier 1' college actually help?

In [27]:
tier_analysis = df.groupby('college_tier')['selected'].mean().reset_index()
tier_analysis['selected_pct'] = tier_analysis['selected'] * 100

fig_tier = px.bar(tier_analysis, x='college_tier', y='selected_pct',
                 title='Selection Rate by College Tier',
                 labels={'selected_pct': 'Selection Rate (%)', 'college_tier': 'College Tier'},
                 color='college_tier',
                 text_auto='.1f')
fig_tier.show()

### Analysis: 
Measures Institutional Equity. If the bars are roughly equal, the selection process is "Prestige-Blind," favoring individual merit over school brand names.

### How to look at it: 
Compare the height of the bars for Tiers 1, 2, and 3.

Question 5: The "Backlog" Threshold
What is the probability of a candidate being selected if there are a couple of backlogs?

In [28]:
backlog_impact = df.groupby('backlogs')['selected'].mean().reset_index()
backlog_impact['success_rate'] = backlog_impact['selected'] * 100

fig_backlog = px.line(backlog_impact, x='backlogs', y='success_rate',
                      title='The "Backlog Threshold": Success Rate vs. Number of Backlogs',
                      markers=True,
                      labels={'success_rate': 'Probability of Selection (%)'})
fig_backlog.show()

### Analysis: 
This identifies the "Red Flag" Threshold. It tells students exactly how many backlogs they can have before their chances of an internship become nearly impossible.

### How to look at it: 
Find the point where the line drops sharply toward 0%.

Question 6: Success Indicators (Mean & Median)
Identify the exact "Target Scores" for successful candidates across key metrics.

In [29]:
# Compare the averages of successful vs. unsuccessful candidates
metrics = ['interview_score', 'coding_test_score', 'skills_score', 'aptitude_score']
success_profile = df.groupby('selected')[metrics].mean().reset_index()

# Reshape for plotting
success_profile_melted = success_profile.melt(id_vars='selected', var_name='Metric', value_name='Average Score')

fig6 = px.bar(success_profile_melted, x='Metric', y='Average Score', 
             color='selected', barmode='group',
             title='Average Performance Scores: Selected vs. Rejected Students',
             color_discrete_map={0: 'lightcoral', 1: 'royalblue'})
fig6.show()

### Analysis: 
This identifies the Key Differentiator. The wider the gap, the more that specific skill (like Interview or Coding) determines the final decision.

### How to look at it:
Look for the skill with the largest difference in height between the "Selected" and "Rejected" bars.

Question 7: Qualitative Synergy
Do "Soft Skills" and "Communication" act as a multiplier for success?

In [30]:
# Creating a combined 'Soft Skills' metric to see its relationship with selection
df['total_soft_skills'] = df['soft_skills_score'] + df['communication_score']

fig7 = px.histogram(df, x='total_soft_skills', color='selected',
                   marginal='violin', 
                   title='Synergistic Effect of Soft Skills on Selection Outcome',
                   labels={'total_soft_skills': 'Combined Communication & Soft Skills Score'})
fig7.show()

### Analysis:
Demonstrates the Behavioral Baseline. It shows that while technical skills get you noticed, high communication scores are often the "Gatekeeper" to being hired.

### How to look at it: 
See if the "Selected" group is shifted far to the right of the "Rejected" group.

Question 8: Practical Experience & Diminishing Returns. Is there a point where adding more projects or internships stops helping?

In [32]:
exp_impact = df.groupby('internships_done')['selected'].mean().reset_index()

fig8 = px.scatter(exp_impact, x='internships_done', y='selected', 
                 size='selected', 
                 title='The Impact of Prior Internships on Selection Probability',
                 labels={'selected': 'Success Rate (0 to 1)', 'internships_done': 'Number of Internships'},
                 trendline="lowess",
                 color='selected',
                 color_continuous_scale='Plasma')

fig8.update_traces(marker=dict(sizemin=10)) 
fig8.show()
print("Selection Probability by Number of Internships:")
print(exp_impact)

Selection Probability by Number of Internships:
   internships_done  selected
0                 0  0.715840
1                 1  0.743508
2                 2  0.732122
3                 3  0.758149


### Analysis: 
Reveals Diminishing Returns. It helps students manage their time by showing if having "some" experience is enough, or if "more" is always better.

### How to look at it: 
Look for where the bars stop getting significantly taller (e.g., from 2 to 3 internships).

Question 9: The "Offset" (Interview vs. CGPA)
Can personality (Interview) save a student with lower grades (CGPA)?

In [39]:
fig9 = px.density_heatmap(df, x="CGPA", y="interview_score", z="selected", 
                         histfunc="avg", 
                         nbinsx=10, nbinsy=10,
                         title="Selection Probability: CGPA vs. Interview Score",
                         labels={'selected': 'Prob of Selection'},
                         color_continuous_scale='Viridis',
                         text_auto='.2f')
fig9.show()

### Analysis: 
This is the "Offset" Proof. If that area is bright/high probability, it proves that a great personality can statistically "save" a student from a lower GPA.

### How to look at it: 
Look at the top-left corner (High Interview + Low CGPA).

Question 10: Behavioral vs. Technical Influence
Which category is the ultimate tie-breaker?

In [40]:
df['Tech_Category'] = np.where(df['Technical_Avg'] >= df['Technical_Avg'].median(), 'High Tech', 'Low Tech')
df['Beh_Category'] = np.where(df['Behavioral_Avg'] >= df['Behavioral_Avg'].median(), 'High Beh', 'Low Beh')

combination_analysis = df.groupby(['Tech_Category', 'Beh_Category'])['selected'].mean().reset_index()
combination_analysis['success_rate'] = combination_analysis['selected'] * 100

fig10 = px.bar(combination_analysis, x='Tech_Category', y='success_rate', 
               color='Beh_Category', barmode='group',
               title='Selection Probability by Technical and Behavioral Skill Level',
               labels={'success_rate': 'Selection Probability (%)'},
               text_auto='.1f')
fig10.show()

### Analysis: 
The Tie-Breaker Analysis. It tells a student exactly where to focus their next month of preparation: on their hard technical skills or their soft behavioral traits.

### How to look at it: 
Compare the "High Tech/Low Beh" bar vs. the "Low Tech/High Beh" bar.